# all-reduce-compose — worked example 3: All_reduce a multi-element gradient vector via reduce + broadcast

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-compose`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The reduce-then-broadcast composition is shape-agnostic: it works element-wise on whole tensors, not just scalars. To all_reduce a gradient vector you wrap the rank-local vector in a tensor, `reduce(SUM, dst=0)` to sum element-wise onto rank 0, then `broadcast(src=0)` to hand the summed vector back to every rank. Each coordinate is reduced independently and in parallel.

## Worked solution

**Step 1 — build the local vector tensor.** `tensor = t.tensor(local_vec, dtype=t.float32)`. Every rank must use the same shape and dtype, or the collective is undefined.

**Step 2 — element-wise reduce to rank 0.** `dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.SUM)` sums position-by-position across ranks; rank 0 ends with `[sum of position 0, sum of position 1, ...]`. Other ranks are stale.

**Step 3 — broadcast the summed vector.** `dist_module.broadcast(tensor, src=0)` copies rank 0's full vector to every rank. Now all ranks hold the identical summed gradient — exactly what data-parallel training needs before the optimizer step.

**Step 4 — return the vector as a list.** `tensor.tolist()` gives the per-coordinate sums, identical on every rank.

**Why it works:** collectives broadcast/reduce the entire tensor buffer, so a vector all_reduce is the same two-step composition as a scalar — the backend just iterates over elements. This is precisely how gradient averaging is built in DDP.

In [ ]:
import threading

class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, world_size):
        self.world_size = world_size
        self._barrier = threading.Barrier(world_size)
        self._slots = [None] * world_size
        self._result = [None]
    def reduce(self, tensor, dst, op):
        rank = threading.current_thread().rank
        self._slots[rank] = tensor.clone()
        self._barrier.wait()
        if rank == dst:
            tensor.copy_(t.stack(self._slots).sum(dim=0))
            self._result[0] = tensor.clone()
        self._barrier.wait()
    def broadcast(self, tensor, src):
        self._barrier.wait()
        tensor.copy_(self._result[0])
        self._barrier.wait()

def ex_all_reduce_vector(rank: int, world_size: int, dist_module, local_vec):
    tensor = t.tensor(local_vec, dtype=t.float32)
    dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.SUM)
    dist_module.broadcast(tensor, src=0)
    return tensor.tolist()

world_size = 3
local_vecs = [[1.0, 2.0, 3.0], [10.0, 20.0, 30.0], [100.0, 200.0, 300.0]]
mock = MockDist(world_size)
results = [None] * world_size

def _run(rank):
    threading.current_thread().rank = rank
    results[rank] = ex_all_reduce_vector(rank, world_size, mock, local_vecs[rank])

threads = [threading.Thread(target=_run, args=(r,)) for r in range(world_size)]
for th in threads: th.start()
for th in threads: th.join()
expected = [sum(col) for col in zip(*local_vecs)]
print('per-rank vectors:', results)
print('all equal:', all(r == results[0] for r in results), '-> vector', results[0], '(expected', expected, ')')